# 🚀 Tái Hiện Thuật Toán LiDAR & RS-LiDAR Bảng 2 Trên Kaggle (2x GPU Tesla T4 16GB)
### **Bài báo**: [Lookahead Sample Reward Guidance for Test-Time Scaling of Diffusion Models (ICML 2026 Spotlight)](https://arxiv.org/abs/2602.03211)

Notebook này được tối ưu hóa toàn diện cho môi trường **Kaggle (2x NVIDIA Tesla T4 16GB VRAM)**:
- **Tối ưu 2x GPU Song Song**: Tận dụng cơ chế Sharding (`--num_shards 2 --shard_id 0/1`) chạy đồng thời trên GPU 0 và GPU 1, tăng tốc gấp đôi và tiết kiệm 50% thời gian thực thi.
- **Tập trung tái lập Bảng 2 & Đối sánh RS-LiDAR**: Hỗ trợ đầy đủ Phase 1 (Lookahead DPM-5, 50 hạt), Phase 2 (LiDAR DDIM-50 hoặc DDPM-100) và Native GenEval Benchmark.
- **Chế độ Fast 2h30 Quota Mode**: Tích hợp `FAST_SAMPLING_MODE = True` chỉ tính ImageReward khi diffusion sampling (tiết kiệm ~50 phút), sau đó chấm offline các metrics còn lại (CLIP-Score, HPS v2.1) trong ~10 phút.
- **Tự Động Khôi Phục Checkpoint**: Quét và tự động nạp lại kết quả từ file `.zip` hoặc Output của phiên Save Version trước trong `/kaggle/input/`.
- **Đóng Gói 1-Click**: Tự động nén toàn bộ kết quả thành `table2_replication_results.zip` tại `/kaggle/working/` để tải về tức thì.

## 1. Kiểm Tra Phần Cứng 2x GPU Tesla T4 & Bộ Nhớ VRAM
Đảm bảo trong cài đặt notebook Kaggle (Cột phải): **Accelerator -> GPU T4 x2**.

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
    if n_gpus >= 2:
        print("✅ Tuyệt vời! Hệ thống đã nhận diện đủ 2x GPU T4 để kích hoạt chế độ chạy song song.")
    else:
        print("⚠️ Lưu ý: Chỉ phát hiện 1 GPU. Vui lòng bật 'GPU T4 x2' ở cột bên phải nếu muốn chạy song song.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings -> Accelerator -> GPU T4 x2.")

!nvidia-smi

## 2. Thiết Lập Thư Mục Làm Việc, Khôi Phục Checkpoint Cũ & Cài Đặt Thư Viện
Tải repository, cài đặt các gói tương thích, giải quyết triệt để các xung đột Protobuf / Transformers, và tải trước trọng số HPSv2.

In [ ]:
import os, sys, shutil, glob, subprocess, threading, urllib.request

# 1. Thiết lập thư mục làm việc trên Kaggle
REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc hiện tại:", os.getcwd())

# Tạo sẵn các thư mục đầu ra chuẩn
os.makedirs(f"{WORKDIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{WORKDIR}/Target_samples", exist_ok=True)

# 2. Tự động khôi phục dữ liệu từ các phiên Save Version trước hoặc file .zip trong /kaggle/input
zip_files = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
for zf in zip_files:
    print(f"📦 Tìm thấy file zip dữ liệu: {zf}. Đang giải nén...")
    try:
        shutil.unpack_archive(zf, WORKDIR)
    except Exception as e:
        print(f"⚠️ Không thể giải nén {zf}: {e}")

for folder in ["Lookahead_samples", "Target_samples"]:
    for src_dir in glob.glob(f"/kaggle/input/**/{folder}/*", recursive=True):
        if os.path.isdir(src_dir):
            base_tag = os.path.basename(src_dir)
            dst_dir = os.path.join(WORKDIR, folder, base_tag)
            os.makedirs(dst_dir, exist_ok=True)
            for p_dir in glob.glob(os.path.join(src_dir, "[0-9]*")):
                p_base = os.path.basename(p_dir)
                p_dst = os.path.join(dst_dir, p_base)
                if not os.path.exists(p_dst) and os.path.isdir(p_dir):
                    shutil.copytree(p_dir, p_dst)

n_look = len(glob.glob(f"{WORKDIR}/Lookahead_samples/*/[0-9]*"))
n_targ = len(glob.glob(f"{WORKDIR}/Target_samples/*/[0-9]*"))
print(f"✅ Đã đồng bộ xong dữ liệu: {n_look} Lookahead prompts, {n_targ} Target prompts sẵn sàng!")

# 3. Hàm tiện ích chạy song song 2 GPU hiển thị log trực tiếp (Real-time Threaded Streaming)
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    p0.wait(); p1.wait()

# 4. Cấu hình biến môi trường chống xung đột và phân mảnh VRAM (Issue 8 & 19)
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 5. Cài đặt các thư viện tương thích (Khắc phục Issue 6, 7, 8, 14)
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 6. Tải trước vocab và checkpoint HPSv2.1 để chống lỗi tải đồng thời trên 2 GPU (Issue 5 & 12)
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
    
    from huggingface_hub import hf_hub_download
    hps_cache = os.path.expanduser("~/.cache/hpsv2")
    os.makedirs(hps_cache, exist_ok=True)
    hps_ckpt = os.path.join(hps_cache, "HPS_v2.1_compressed.pt")
    if not os.path.exists(hps_ckpt) or os.path.getsize(hps_ckpt) < 1000000:
        print("⏳ Đang tải trước trọng số HPSv2.1...")
        try:
            hf_hub_download(repo_id="xswu/HPSv2", filename="HPS_v2.1_compressed.pt", local_dir=hps_cache)
            print("✅ Đã tải xong HPSv2.1!")
        except Exception:
            urllib.request.urlretrieve("https://huggingface.co/xswu/HPSv2/resolve/main/HPS_v2.1_compressed.pt", hps_ckpt)
            print("✅ Đã tải xong HPSv2.1!")
except Exception as e:
    print(f"Lưu ý khởi tạo HPS: {e}")

print("\n✅ Môi trường trên Kaggle đã được cấu hình hoàn hảo 100%!")

## 3. Cấu Hình Siêu Tham Số Thí Nghiệm Chuẩn Bảng 2 & RS-LiDAR
Thiết lập tham số theo đúng Bảng 2 của bài báo gốc ($s=12.5$ hoặc $15.0, \lambda=5000, t_{end}=200, n=50$) trên backbone Stable Diffusion v1.5.

In [ ]:
import glob, os

# ==============================================================================
# 🎯 1. CẤU HÌNH MÔ HÌNH VÀ THIẾT BỊ
# ==============================================================================
NUM_GPUS = 2                     # 2 GPU T4 chạy song song (hoặc 1 nếu chỉ dùng 1 GPU)
SEED = 100                       # Seed cố định (100 chuẩn Bảng 2)
MODEL_NAME = "runwayml/stable-diffusion-v1-5"
MODEL_TAG = "SD15"

# ==============================================================================
# 🔬 2. CẤU HÌNH PHASE 1 (LOOKAHEAD SAMPLING)
# ==============================================================================
NUM_LOOKAHEAD_PARTICLES = 50     # n = 50 hạt
LOOKAHEAD_STEPS = 5              # 5 bước DPM-Solver (DPM-5)

# Cấu hình Randomized Smoothing (RS-LiDAR Phase 1)
USE_SMOOTHING = False            # True = RS-LiDAR, False = Vanilla LiDAR gốc
SIGMA = 1.0                      # Bán kính làm mịn sigma (chuẩn 1.0 cho ảnh decode [-1, 1])
NUM_MC_SAMPLES = 4               # M = 4 mẫu Monte Carlo

# Tên định danh thư mục Lookahead
if USE_SMOOTHING:
    LOOKAHEAD_TAG = f"RSLookahead_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{NUM_LOOKAHEAD_PARTICLES}_sig{SIGMA}_M{NUM_MC_SAMPLES}_seed{SEED}"
else:
    LOOKAHEAD_TAG = f"100_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"

# Tự động tái sử dụng latents giữa Vanilla và RS-LiDAR nếu đã có sẵn
REUSE_LATENTS_FROM = f"100_{NUM_LOOKAHEAD_PARTICLES}_{LOOKAHEAD_STEPS}"

# ==============================================================================
# 🎯 3. CẤU HÌNH PHASE 2 (TARGET SAMPLING)
# ==============================================================================
NUM_TARGET_STEPS = 50            # 50 bước DDIM (hoặc 100 bước DDPM)
ETA = 0.0                        # 0.0 = DDIM, 1.0 = DDPM
TARGET_PARTICLES = 4             # 4 ảnh trên mỗi prompt (chuẩn đánh giá GenEval)
SCALE = 15.0                     # Hệ số guidance s = 15.0 để đạt ImageReward 0.378 theo Bảng 2
LAMBDA = 5000.0                  # Hệ số lambda = 5000
RESAMPLE_T_END = 200             # Ngưỡng kết thúc guidance sớm
TOP_K = 50                       # Top-k lookaheads

# Tự động sinh tên thư mục chạy Phase 2
SOLVER_TAG = f"DDPM{NUM_TARGET_STEPS}" if ETA == 1.0 else (f"DDIM{NUM_TARGET_STEPS}" if ETA == 0.0 else f"DDIM{NUM_TARGET_STEPS}_eta{ETA}")
if USE_SMOOTHING:
    RUN_NAME = f"RSLiDAR_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{NUM_LOOKAHEAD_PARTICLES}_sig{SIGMA}_M{NUM_MC_SAMPLES}_{SOLVER_TAG}_s{SCALE}_lmbda{int(LAMBDA)}_seed{SEED}"
else:
    RUN_NAME = f"LiDAR_{MODEL_TAG}_DPM{LOOKAHEAD_STEPS}_n{NUM_LOOKAHEAD_PARTICLES}_{SOLVER_TAG}_s{SCALE}_lmbda{int(LAMBDA)}_seed{SEED}"

# Dữ liệu Prompt và Giới hạn số lượng (thử nghiệm: 10, toàn bộ Bảng 2: 553)
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"
MAX_PROMPTS = 553

# ==============================================================================
# ⚡ 4. CÔNG TẮC TỐC ĐỘ CAO (FAST 2H30 MODE)
# ==============================================================================
FAST_SAMPLING_MODE = True        # True: Chỉ tính ImageReward khi diffusion (tiết kiệm ~50 phút), sau đó chấm bổ sung ở Mục 6

lookahead_dir = f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}"
target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"

existing_look = len(glob.glob(f"{lookahead_dir}/[0-9]*/results.json"))
existing_targ = len(glob.glob(f"{target_dir}/[0-9]*/results.json"))

print("=" * 75)
print(f"📊 TIẾN ĐỘ THỰC NGHIỆM TRÊN KAGGLE ({NUM_GPUS}x GPU T4):")
print(f"  • Cấu hình: {MODEL_TAG} | Solver: {SOLVER_TAG} (eta={ETA}) | Scale: {SCALE}")
print(f"  • Chế độ Smoothing: {USE_SMOOTHING} (sigma={SIGMA}, M={NUM_MC_SAMPLES})")
print(f"  • Thư mục Phase 1: {LOOKAHEAD_TAG} (Hoàn thành: {existing_look}/{MAX_PROMPTS})")
print(f"  • Thư mục Phase 2: {RUN_NAME} (Hoàn thành: {existing_targ}/{MAX_PROMPTS})")
print("=" * 75)

## 4. Phase 1: Lấy Mẫu Lookahead Song Song Trên 2 GPU (Lookahead Sampling)
Chạy bộ giải DPM-5 sinh 50 hạt. Nếu bật 2 GPU, tập prompt sẽ được chia đều 50/50 cho GPU 0 và GPU 1.

In [ ]:
import os, glob
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

existing_look = len(glob.glob(f"{WORKDIR}/Lookahead_samples/{LOOKAHEAD_TAG}/[0-9]*/results.json"))

if existing_look >= MAX_PROMPTS:
    print(f"⏩ [SKIP PHASE 1] Đã hoàn thành đủ {existing_look}/{MAX_PROMPTS} prompt Lookahead! Chuyển thẳng sang Phase 2.")
else:
    # 0. Tiền tải weights vào cache ổ đĩa 1 lần bằng huggingface_hub (hoàn toàn không import diffusers trong notebook kernel)
    print("📥 Đang kiểm tra / tiền tải weights vào cache ổ đĩa...")
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL_NAME)

    ir_weight_path = os.path.expanduser("~/.cache/ImageReward/ImageReward.pt")
    if not os.path.exists(ir_weight_path):
        os.makedirs(os.path.dirname(ir_weight_path), exist_ok=True)
        import urllib.request
        try:
            urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/ImageReward.pt", ir_weight_path)
            urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/med_config.json", os.path.join(os.path.dirname(ir_weight_path), "med_config.json"))
        except Exception as e:
            print(f"Lưu ý: Không thể tải trước ImageReward ({e}), ImageReward sẽ tự động tải khi chạy.")
    print("✅ Weights đã có sẵn trên đĩa! Bắt đầu thực thi song song 2 GPU hoàn toàn không bị nghẽn mạng hay đợi model.")

    smooth_args = f"--use_smoothing --sigma={SIGMA} --num_mc_samples={NUM_MC_SAMPLES}" if USE_SMOOTHING else ""
    reuse_args = f"--reuse_latents_from={REUSE_LATENTS_FROM}" if (USE_SMOOTHING and os.path.exists(f"{WORKDIR}/Lookahead_samples/{REUSE_LATENTS_FROM}")) else ""
    
    if NUM_GPUS == 2:
        print(f"🚀 [2 GPU] Đang chạy song song Phase 1 trên GPU 0 và GPU 1 ({MAX_PROMPTS} prompts)...")
        cmd0 = f"""python -u lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --run_name="{LOOKAHEAD_TAG}" \
            --gpu_id=0 --num_shards=2 --shard_id=0 --resume {smooth_args} {reuse_args}"""
            
        cmd1 = f"""python -u lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --run_name="{LOOKAHEAD_TAG}" \
            --gpu_id=1 --num_shards=2 --shard_id=1 --resume {smooth_args} {reuse_args}"""
            
        run_commands_parallel(cmd0, cmd1)
        print("✅ Cả 2 GPU đã hoàn thành Giai đoạn 1!")
    else:
        print(f"🚀 [1 GPU] Đang chạy Phase 1 trên 1 GPU ({MAX_PROMPTS} prompts)...")
        !python -u lookahead_sampling.py \
            --seed={SEED} --num_particles={NUM_LOOKAHEAD_PARTICLES} --num_inference_steps={LOOKAHEAD_STEPS} \
            --model_name="{MODEL_NAME}" --guidance_reward_fn="ImageReward" --metrics_to_compute="ImageReward#Clip-Score" \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --run_name="{LOOKAHEAD_TAG}" \
            --resume {smooth_args} {reuse_args}

## 5. Phase 2: Lấy Mẫu Đích LiDAR Sampling Song Song Trên 2 GPU
Thực thi thuật toán dẫn đường closed-form để sinh 4 ảnh trên mỗi prompt, chia tải song song trên 2x GPU T4.

In [ ]:
import os, glob
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.chdir(WORKDIR)

existing_targ = len(glob.glob(f"{WORKDIR}/Target_samples/{RUN_NAME}/[0-9]*/results.json"))

if existing_targ >= MAX_PROMPTS:
    print(f"⏩ [SKIP PHASE 2] Đã hoàn thành đủ {existing_targ}/{MAX_PROMPTS} prompt Target Sampling! Chuyển sang đánh giá.")
else:
    metrics_arg = '--metrics_to_compute="ImageReward"' if FAST_SAMPLING_MODE else '--metrics_to_compute="ImageReward#Clip-Score#HumanPreference#Clip-Diversity#AS"'
    
    if NUM_GPUS == 2:
        print(f"🚀 [2 GPU] Đang chạy song song Phase 2 trên GPU 0 và GPU 1 (Run: {RUN_NAME})...")
        cmd0 = f"""python -u LiDAR_sampling.py \
            --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
            --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
            {metrics_arg} \
            --save_individual_images --run_name="{RUN_NAME}" --gpu_id=0 --num_shards=2 --shard_id=0 --resume"""
            
        cmd1 = f"""python -u LiDAR_sampling.py \
            --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
            --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
            {metrics_arg} \
            --save_individual_images --run_name="{RUN_NAME}" --gpu_id=1 --num_shards=2 --shard_id=1 --resume"""
            
        run_commands_parallel(cmd0, cmd1)
        print("✅ Cả 2 GPU đã hoàn thành Giai đoạn 2!")
    else:
        print(f"🚀 [1 GPU] Đang chạy Phase 2 trên 1 GPU (Run: {RUN_NAME})...")
        !python -u LiDAR_sampling.py \
            --seed={SEED} --model_name="{MODEL_NAME}" --num_particles={TARGET_PARTICLES} --num_inference_steps={NUM_TARGET_STEPS} \
            --eta={ETA} --use_rag --lookahead_path="{LOOKAHEAD_TAG}" --top_k={TOP_K} --scale={SCALE} --lmbda={LAMBDA} --resample_t_end={RESAMPLE_T_END} \
            --prompt_path="{PROMPT_FILE}" --max_prompt={MAX_PROMPTS} --guidance_reward_fn="ImageReward" \
            {metrics_arg} \
            --save_individual_images --run_name="{RUN_NAME}" --resume

## 6. Đánh Giá Offline Bổ Sung Nhanh Các Metrics Cho Bảng 2
Nếu bật `FAST_SAMPLING_MODE`, cell này tự động quét các ảnh đã lưu để chấm điểm bổ sung **CLIP-Score, HPS v2.1, Aesthetic Score, Diversity** vào `results.json` chỉ trong ~8-12 phút.

In [ ]:
import os, glob, json, sys, gc, torch
from PIL import Image
from tqdm import tqdm

if f"{WORKDIR}/fkd_diffusers" not in sys.path: sys.path.insert(0, f"{WORKDIR}/fkd_diffusers")
if WORKDIR not in sys.path: sys.path.insert(0, WORKDIR)

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

from fks_utils import do_eval

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
prompt_dirs = sorted(glob.glob(f"{target_dir}/[0-9]*"))
print(f"🔍 Đang kiểm tra chỉ số cho {len(prompt_dirs)} prompt trong {target_dir}...")

needed_metrics = ["Clip-Score", "HumanPreference", "Clip-Diversity", "AS"]
prompts_to_eval = []

for p_dir in prompt_dirs:
    res_file = os.path.join(p_dir, "results.json")
    meta_file = os.path.join(p_dir, "metadata.jsonl")
    if not os.path.exists(res_file) or not os.path.exists(meta_file): continue
    try:
        with open(res_file, "r", encoding="utf-8") as f: res_data = json.load(f)
    except Exception: res_data = {}
    missing = [m for m in needed_metrics if m not in res_data]
    if missing: prompts_to_eval.append((p_dir, res_file, meta_file, missing))

if not prompts_to_eval:
    print("✅ Toàn bộ prompt đã có đầy đủ chỉ số đánh giá! Sẵn sàng lập Bảng 2.")
else:
    print(f"⏳ Cần chấm bổ sung ({', '.join(needed_metrics)}) cho {len(prompts_to_eval)} prompt...")
    for p_dir, res_file, meta_file, missing in tqdm(prompts_to_eval, desc="Đánh giá bổ sung CLIP & HPS"):
        try:
            with open(meta_file, "r", encoding="utf-8") as f: meta = json.load(f)
            p_str = meta.get("prompt", meta.get("text", ""))
            img_files = sorted(glob.glob(f"{p_dir}/samples/*.png"))
            if not img_files:
                img_files = sorted([f for f in glob.glob(f"{p_dir}/*.png") if not f.endswith("grid.png")])
            if not img_files: continue
            imgs = [Image.open(p).convert("RGB") for p in img_files]
            with torch.inference_mode():
                new_res = do_eval(prompt=[p_str] * len(imgs), images=imgs, metrics_to_compute=missing)
            with open(res_file, "r", encoding="utf-8") as f: curr_res = json.load(f)
            for m in missing:
                if m in new_res: curr_res[m] = new_res[m]
            with open(res_file, "w", encoding="utf-8") as f: json.dump(curr_res, f, indent=4)
        except Exception as e:
            print(f"⚠️ Lỗi chấm bổ sung tại prompt {p_dir}: {e}")
    print("\n🎉 Hoàn tất đánh giá bổ sung! Toàn bộ kết quả đã được ghi vào results.json.")

## 7. Đánh Giá Chuẩn Mực GenEval Benchmark (Native PyTorch & HuggingFace)
Chấm điểm 6 nhiệm vụ của GenEval (single object, two objects, counting, colors, position, color attribution) bằng Faster R-CNN + CLIP zero-shot (khắc phục hoàn toàn lỗi MMDetection C++ cũ).

In [ ]:
import os, glob, json, urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm

from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as TF
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Thiết bị tính toán GenEval: {device}")

metadata_candidates = glob.glob(f"{WORKDIR}/**/geneval_metadata.jsonl", recursive=True) + glob.glob("/kaggle/**/geneval_metadata.jsonl", recursive=True)
if metadata_candidates and os.path.exists(metadata_candidates[0]):
    metadata_path = metadata_candidates[0]
else:
    metadata_path = f"{WORKDIR}/prompt_files/geneval_metadata.jsonl"
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl", metadata_path)

prompts_meta = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): prompts_meta.append(json.loads(line.strip()))
print(f"📋 Đã nạp {len(prompts_meta)} prompt chuẩn GenEval.")

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
prompt_dir_map = {}
for p_dir in glob.glob(f"{target_dir}/[0-9]*"):
    try: p_idx = int(os.path.basename(p_dir))
    except ValueError: continue
    imgs = sorted(glob.glob(f"{p_dir}/samples/*.png"))
    if not imgs: imgs = sorted([img for img in glob.glob(f"{p_dir}/*.png") if not img.endswith("grid.png")])
    if imgs: prompt_dir_map[p_idx] = imgs

print(f"🎯 Đã tìm thấy {len(prompt_dir_map)} prompts có ảnh để chấm điểm.")

overall_geneval = None
if prompt_dir_map:
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    detector = fasterrcnn_resnet50_fpn(weights=weights).to(device).eval()
    coco_classes = weights.meta["categories"]
    
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    COLORS = ["red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "black", "white"]
    color_prompts = [f"a photo of a {c} object" for c in COLORS]
    
    def classify_crop_color(crop_img):
        inputs = clip_processor(text=color_prompts, images=crop_img, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = clip_model(**inputs)
            best_idx = outputs.logits_per_image.argmax(dim=-1).item()
            return COLORS[best_idx]
    
    task_results = {"single_object": [], "two_object": [], "counting": [], "colors": [], "position": [], "color_attr": []}
    for p_idx in tqdm(sorted(prompt_dir_map.keys()), desc="Chấm điểm GenEval"):
        if p_idx >= len(prompts_meta): continue
        meta = prompts_meta[p_idx]
        tag = meta.get("tag", "single_object")
        if tag not in task_results: continue
        
        images_paths = prompt_dir_map[p_idx]
        prompt_scores = []
        for img_path in images_paths:
            try:
                img = Image.open(img_path).convert("RGB")
                img_t = TF.to_tensor(img).to(device)
                with torch.no_grad(): preds = detector([img_t])[0]
                
                scores, labels, boxes = preds["scores"].cpu().numpy(), preds["labels"].cpu().numpy(), preds["boxes"].cpu().numpy()
                keep = scores > 0.35
                detected_labels, detected_boxes = labels[keep], boxes[keep]
                
                detected_objects = []
                for lbl, box in zip(detected_labels, detected_boxes):
                    c_name = coco_classes[lbl].lower()
                    x1, y1, x2, y2 = box
                    if (x2 - x1 > 12 and y2 - y1 > 12):
                        crop = img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))
                        pred_color = classify_crop_color(crop)
                    else:
                        pred_color = "unknown"
                    detected_objects.append({"class": c_name, "box": box, "center_x": (x1+x2)/2.0, "center_y": (y1+y2)/2.0, "color": pred_color})
                
                success = False
                includes = meta.get("include", [])
                if tag == "single_object":
                    req_cls = includes[0]["class"].lower()
                    success = any(req_cls in obj["class"] or obj["class"] in req_cls for obj in detected_objects)
                elif tag == "two_object":
                    req1, req2 = includes[0]["class"].lower(), includes[1]["class"].lower()
                    success = any(req1 in obj["class"] or obj["class"] in req1 for obj in detected_objects) and any(req2 in obj["class"] or obj["class"] in req2 for obj in detected_objects)
                elif tag == "counting":
                    req_cls, target_count = includes[0]["class"].lower(), includes[0]["count"]
                    found_count = sum(1 for obj in detected_objects if req_cls in obj["class"] or obj["class"] in req_cls)
                    success = (found_count == target_count)
                elif tag == "colors":
                    req_cls, req_color = includes[0]["class"].lower(), includes[0]["color"].lower()
                    success = any((req_cls in obj["class"] or obj["class"] in req_cls) and (obj["color"] == req_color) for obj in detected_objects)
                elif tag == "position":
                    req1, req2 = includes[0]["class"].lower(), includes[1]["class"].lower()
                    pos_type = includes[1].get("position", ["right of", 0])[0]
                    o1_list = [o for o in detected_objects if req1 in o["class"] or o["class"] in req1]
                    o2_list = [o for o in detected_objects if req2 in o["class"] or o["class"] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        if "right" in pos_type: success = (o2["center_x"] > o1["center_x"])
                        elif "left" in pos_type: success = (o2["center_x"] < o1["center_x"])
                        elif "above" in pos_type or "top" in pos_type: success = (o2["center_y"] < o1["center_y"])
                        elif "below" in pos_type or "bottom" in pos_type: success = (o2["center_y"] > o1["center_y"])
                        else: success = True
                elif tag == "color_attr":
                    success = all(any((inc["class"].lower() in obj["class"] or obj["class"] in inc["class"].lower()) and (obj["color"] == inc["color"].lower()) for obj in detected_objects) for inc in includes)
                
                prompt_scores.append(1.0 if success else 0.0)
            except Exception:
                pass
        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))
    
    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({"Nhiệm Vụ (Task)": t_name, "Số Prompt": len(scores), "Độ Chính Xác (Accuracy ↑)": f"{mean_val:.4f}"})
    
    overall_geneval = np.mean(all_means) if all_means else 0.0
    summary_rows.append({"Nhiệm Vụ (Task)": "🔥 OVERALL GENEVAL BENCHMARK", "Số Prompt": sum(len(s) for s in task_results.values()), "Độ Chính Xác (Accuracy ↑)": f"{overall_geneval:.4f}"})
    df_geneval = pd.DataFrame(summary_rows)
    print("\n" + "="*70)
    print("📊 BẢNG TỔNG HỢP ĐIỂM SỐ GENEVAL BENCHMARK")
    print("="*70)
    print(df_geneval.to_string(index=False))
    print("="*70)
    df_geneval.to_csv(f"{target_dir}/geneval_summary.csv", index=False)
    df_geneval.to_csv("/kaggle/working/geneval_summary.csv", index=False)

## 8. Bảng Đối Chiếu Kết Quả Toàn Diện Với Bảng 2 (ICML 2026)
Tổng hợp đầy đủ 4 cột chỉ số chuẩn Bảng 2: **ImageReward, CLIP-Score, HPS v2.1, và GenEval**.

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
from IPython.display import display

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
result_files = sorted(glob.glob(f"{target_dir}/[0-9]*/results.json"))

if result_files:
    print(f"📊 Đang tổng hợp chỉ số đánh giá từ {len(result_files)} prompt...")
    metric_keys = ["ImageReward", "Clip-Score", "HumanPreference", "Clip-Diversity", "AS"]
    collected_means = {k: [] for k in metric_keys}
    
    for rf in result_files:
        try:
            with open(rf, "r") as f: res = json.load(f)
            for k in metric_keys:
                if k in res and "mean" in res[k]:
                    collected_means[k].append(res[k]["mean"])
        except Exception:
            pass
    
    final_metrics = {}
    for k, vals in collected_means.items():
        if vals:
            final_metrics[k] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    
    if 'overall_geneval' in locals() and overall_geneval is not None:
        final_metrics["GenEval"] = {"mean": float(overall_geneval)}
    
    with open(f"{target_dir}/final_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    
    ir_val = final_metrics.get('ImageReward', {}).get('mean', 0.0)
    clip_val = final_metrics.get('Clip-Score', {}).get('mean', 0.0)
    hps_val = final_metrics.get('HumanPreference', {}).get('mean', 0.0)
    ge_val = final_metrics.get('GenEval', {}).get('mean', None)
    ge_str = f"{ge_val:.4f}" if ge_val is not None else "Chưa tính"
    
    table_data = [
        {"Phương Pháp": "SD v1.5 Gốc (DDIM-50)", "Số Bước": "50 DDIM", "ImageReward ↑": "-0.125", "CLIP-Score ↑": "0.269", "HPS v2.1 ↑": "0.270", "GenEval ↑": "0.423", "Đánh Giá": "Baseline"},
        {"Phương Pháp": "SD v1.5 Gốc (DDPM-100)", "Số Bước": "100 DDPM", "ImageReward ↑": "0.001", "CLIP-Score ↑": "0.271", "HPS v2.1 ↑": "0.263", "GenEval ↑": "0.426", "Đánh Giá": "Baseline"},
        {"Phương Pháp": "LiDAR Bảng 2 (DDIM-50)", "Số Bước": "50 DDIM", "ImageReward ↑": "0.378", "CLIP-Score ↑": "0.278", "HPS v2.1 ↑": "0.277", "GenEval ↑": "0.475", "Đánh Giá": "Paper Target"},
        {"Phương Pháp": "LiDAR Bảng 2 (DDPM-100)", "Số Bước": "100 DDPM", "ImageReward ↑": "0.384", "CLIP-Score ↑": "0.278", "HPS v2.1 ↑": "0.276", "GenEval ↑": "0.478", "Đánh Giá": "Paper Upper Bound"},
        {"Phương Pháp": f"🔥 HIỆN TẠI (Ours {'+ RS' if USE_SMOOTHING else ''})", "Số Bước": f"{NUM_TARGET_STEPS} {SOLVER_TAG}", "ImageReward ↑": f"{ir_val:.4f}", "CLIP-Score ↑": f"{clip_val:.4f}", "HPS v2.1 ↑": f"{hps_val:.4f}", "GenEval ↑": ge_str, "Đánh Giá": f"Δ IR: {ir_val - 0.378:+.3f}"}
    ]
    
    df = pd.DataFrame(table_data)
    print("\n======================= 📊 BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BÀI BÁO (BẢNG 2) =======================")
    display(df)
    
    df.to_csv("/kaggle/working/table2_comparison.csv", index=False)
    df.to_csv(f"{target_dir}/table2_comparison.csv", index=False)
    with open("/kaggle/working/table2_comparison.md", "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))
    with open("/kaggle/working/final_metrics.json", "w", encoding="utf-8") as f:
        json.dump(final_metrics, f, indent=4)
    print("\n💾 Đã tự động xuất bảng kết quả đầy đủ (kèm GenEval) ra file:")
    print("  • CSV: /kaggle/working/table2_comparison.csv")
    print("  • Markdown: /kaggle/working/table2_comparison.md")
    print("  • JSON: /kaggle/working/final_metrics.json")

## 9. Trực Quan Hóa Lưới Ảnh Mẫu Đã Sinh
Hiển thị trực quan 3 prompt đầu tiên để kiểm tra chất lượng ảnh sinh ra.

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
grid_images = sorted(glob.glob(f"{target_dir}/*/grid.png"))

if grid_images:
    print(f"🖼️ Tìm thấy {len(grid_images)} lưới ảnh. Đang hiển thị tối đa 3 prompt:")
    for img_path in grid_images[:3]:
        img = Image.open(img_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Prompt: {os.path.basename(os.path.dirname(img_path))}")
        plt.show()
else:
    print("Chưa tìm thấy ảnh lưới mẫu nào.")

## 10. Đóng Gói Tự Động Toàn Bộ Kết Quả (1-Click Download)
Nén toàn bộ ảnh mẫu, latents, file JSON và báo cáo CSV thành `table2_replication_results.zip` tại `/kaggle/working/` để bạn có thể tải về dễ dàng từ tab **Output** của Kaggle.

In [ ]:
import os, shutil

zip_name = "/kaggle/working/table2_replication_results.zip"
print(f"📦 Đang nén dữ liệu kết quả vào {zip_name}...")

# Thu thập các thư mục và file quan trọng
!zip -r -q {zip_name} \
    Target_samples/{RUN_NAME} \
    Lookahead_samples/{LOOKAHEAD_TAG} \
    /kaggle/working/*.csv \
    /kaggle/working/*.md \
    /kaggle/working/*.json 2>/dev/null || true

if os.path.exists(zip_name):
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"📁 Đường dẫn file zip: {zip_name}")
    print("👉 Bạn có thể tải file này từ tab 'Output' của giao diện Kaggle hoặc liên kết sang Dataset tiếp theo!")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")